In [1]:
# Import everything
import numpy as np
import torch
import torch.nn.functional as F
import wandb
import os
import matplotlib.pyplot as plt


import sys

# Add parent directory to Python path so we can import `models.*`
sys.path.append(os.path.abspath(".."))


from models.text_encoder import TextEncoder
from models.img_encoder import VisionEncoder
from models.shared_head import SharedHead
from metrics.retrieval import retrieval
from analysis.modality_gap import compute_gap

from dataloader import get_dataloaders
from pipelines import train_model_with_visualization, test_model_against_tasks
from config_loader import load_configs_from_dir
from tqdm import tqdm
import glob
import yaml
from typing import List, Tuple
from config import Config
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import open_clip

[nltk_data] Downloading package punkt to /home/emanuele/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/emanuele/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/emanuele/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
def load_configs_from_dir(config_dir: str) -> List[Tuple[str, Config]]:
    paths = sorted(glob.glob(os.path.join(config_dir, "*.yaml")))
    if not paths:
        raise FileNotFoundError(f"No .yaml configs found in: {config_dir}")

    configs = []
    for p in paths:
        with open(p, "r", encoding="utf-8") as f:
            d = yaml.safe_load(f) or {}
        cf = Config.from_dict(d)
        configs.append((p, cf))
    return configs

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Numpy original from https://github.com/tulip-control/polytope/blob/main/polytope/polytope.py
# Converted to PyTorch by SHH

import torch
import math
import torch.nn as nn


def givens_rotation_matrix(i, j, theta, N):
    """Return the Givens rotation matrix for an N-dimensional space."""
    R = torch.eye(N)
    c = torch.cos(theta)
    s = torch.sin(theta)
    R[i, i] = c
    R[j, j] = c
    R[i, j] = -s
    R[j, i] = s
    return R

def solve_rotation_ap(u, v, check_vecs=False, debug=True):
    """Return the rotation matrix for the rotation in the plane defined by the
    vectors u and v across TWICE the angle between u and v.

    This algorithm uses the Aguilera-Perez Algorithm \cite{Aguilera} (https://dspace5.zcu.cz/bitstream/11025/6178/1/N29.pdf)
    to generate the rotation matrix. The algorithm works basically as follows:

    Starting with the Nth component of u, rotate u towards the (N-1)th
    component until the Nth component is zero. Continue until u is parallel to
    the 0th basis vector. Next do the same with v until it only has none zero
    components in the first two dimensions. The result will be something like
    this:

    [[u0,  0, 0 ... 0],
     [v0, v1, 0 ... 0]]

    Now it is trivial to align u with v. Apply the inverse rotations to return
    to the original orientation.

    NOTE: The precision of this method is limited by sin, cos, and arctan
    functions.
    Also NOTE: Reversing order of u,v -> v,u yields R.T
    """
    u, v = u.squeeze(), v.squeeze()
    assert len(u.shape)==1, f"u ({list(u.shape)}) & v ({list(v.shape)}) should be single vectors, not batches"

    # BTW: pretty safe to assume u & v have same dims, on same device
    device = u.device
    N = len(u)                       # the number of dimensions
    M = torch.eye(N).to(device)      # stores rotation matrix

    # optional: maybe save a bit of time for (anti-)parallel or zero u & v
    if check_vecs and u.norm() * v.norm() == torch.dot(u,v).abs(): 
        if debug:
            print(f"solve_rotation_ap: zero or (anti-)parallel u,v: 0 degree rotation")
        return M 

    uv = torch.stack([u, v], axis=1)  # the plane of rotation
    # ensure u has positive basis0 component
    if uv[0, 0] < 0:
        M[0, 0] = -1
        M[1, 1] = -1
        uv = M.matmul(uv)
    # align uv plane with the basis01 plane and u with basis0.
    for c in range(2):
        for r in range(N - 1, c, -1):
            if uv[r, c] != 0:  # skip rotations when theta will be zero
                theta = torch.arctan2(uv[r, c], uv[r - 1, c])
                Mk = givens_rotation_matrix(r, r - 1, theta, N).to(device)
                uv = Mk.matmul(uv)
                M = Mk.matmul(M)
    # rotate u onto v
    theta = 2 * torch.arctan2(uv[1, 1], uv[0, 1])
    if debug:
        print(f"solve_rotation_ap: {180 * theta / math.pi:6.2f} degree rotation")
    R = givens_rotation_matrix(0, 1, theta, N).to(device)
    # perform M rotations in reverse order
    M_inverse = M.T
    R = M_inverse.matmul(R.matmul(M))
    return R

def rotate_batch(R, v_batch):
    #return v_batch @ R  # we could multiply from the right (aka "passive rotations")
    #^But conventionally, ("active") rotations are applied by multiplying from the left, so...
    #return (R @ v_batch.T).T      #  ...but (A @ B).T = B.T @ A.T, so then...
    return v_batch @ R.T   # linear algebra FTW

def get_rot_2d(theta, nd=2):
    c, s = torch.cos(theta), torch.sin(theta)
    return torch.tensor([ [c, -s],[s,c] ])

def get_rot_nd(u, v, debug=False):
    """Return the rotation matrix that rotates u onto v."""
    return solve_rotation_ap(u, v, debug=debug)

class FiLMR2d(nn.Module):
    "affine transformation plus rotation, in 2d"
    def __init__(self, nd=2, 
                 beta_init_fac = 0.0001, # tiny beta for FILM is maybe cheating
                 theta_init_fac=6.28/15, # 2pi/n init is "cheating" a bit
                 ):
        super().__init__()
        self.gamma =  nn.Parameter(torch.ones((1)))
        self.beta =  beta_init_fac*nn.Parameter(torch.randn((1)))
        self.theta = theta_init_fac*nn.Parameter( torch.ones((1)) ) 

    def forward(self, x):
        rot = get_rot(self.theta).to(x.device)
        return (x * self.gamma + self.beta.to(x.device)) @ rot

class FiLMRnd(nn.Module):
    "affine transformation plus rotation, in nd"
    def __init__(self, nd=3, 
                 beta_init_fac = 0.0001, # tiny beta is maybe cheating
                 uv_diff_fac = 0.25, # difference scale between initial u and v
                 norm_uv = False,
                 use_bivector=True,
                 rank_uv=1, # just an option I tried. leave it at 1
                 ):
        super().__init__()
        self.norm_uv, self.use_bivector = norm_uv, use_bivector
        self.gamma =  nn.Parameter(torch.ones((1)))
        self.beta =  beta_init_fac*nn.Parameter(torch.randn((1)))
        #self.theta = theta_init_fac*nn.Parameter( torch.ones((1)) ) 
        
        self.u = torch.randn((rank_uv,nd)) 
        self.v = self.u + uv_diff_fac*torch.randn((rank_uv,nd))   # make v a bit different from u
        if self.norm_uv:
            # FYI: normalizing u & v, either initially or at all times, doesn't magically fix everything but we can try it
            self.u = self.u / self.u.norm(dim=-1)
            self.v = self.v / self.v.norm(dim=-1)
        self.u = nn.Parameter( self.u )
        self.v = nn.Parameter( self.v ) 

    def forward(self, x, debug=False):
        u,v = self.u, self.v
        if self.use_bivector:
            if len(u.shape)<2:
                u, v = u.unsqueeze(0), v.unsqueeze(0)
            rot = 0.5 * ( u.T @ v - v.T @ u)
        else:
            #rot = get_rot(self.theta).to(x.device)
            rot = get_rot_nd(self.u, self.v, debug=debug).to(x.device)
        if debug: print("self.u.shape, self.v.shape, rot.shape =",self.u.shape, self.v.shape,rot.shape)
        return (x * self.gamma + self.beta.to(x.device)) @ rot


In [4]:
gaps = ['RMG','L2M','L2I']


In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import open_clip
from tqdm.auto import tqdm


# -------------------------
# 1) Minimal CLIP + FiLMR wrapper
# -------------------------
class CLIPFiLMRMapper(nn.Module):
    """
    Map CLIP image embeddings -> CLIP text embeddings using FiLMR.
    Encoders are frozen; only FiLMR is trained.
    """
    def __init__(self, clip_model: nn.Module, embed_dim: int):
        super().__init__()
        self.clip = clip_model
        self.mapper = FiLMRnd(nd=embed_dim)  # your FiLMR layer

        # freeze CLIP
        for p in self.clip.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def encode_image(self, images):
        z = self.clip.encode_image(images)
        return F.normalize(z, dim=-1)

    @torch.no_grad()
    def encode_text(self, tokens):
        z = self.clip.encode_text(tokens)
        return F.normalize(z, dim=-1)

    def forward(self, images, text_tokens):
        # frozen targets
        with torch.no_grad():
            z_img = self.encode_image(images)          # [B, D]
            z_txt = self.encode_text(text_tokens)      # [B, D]

        # trainable mapping
        z_hat = self.mapper(z_img)                     # [B, D] (per your impl)
        z_hat = F.normalize(z_hat, dim=-1)

        return z_hat, z_txt


# -------------------------
# 2) Loss options
# -------------------------
def cosine_regression_loss(z_hat, z_target):
    # 1 - cosine similarity (mean over batch)
    return (1.0 - (z_hat * z_target).sum(dim=-1)).mean()

def contrastive_loss(z_hat, z_target, temperature=0.07):
    """
    CLIP-style batch contrastive: match each z_hat[i] to z_target[i].
    """
    logits = (z_hat @ z_target.t()) / temperature  # [B,B]
    labels = torch.arange(z_hat.size(0), device=z_hat.device)
    return F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels)


# -------------------------
# 3) Minimal training loop
# -------------------------
def train_filmr_mapper(
    cf,
    dataloader: DataLoader,
    device="cuda",
    model_name="ViT-B-32",
    # pretrained=None,
    pretrained="laion2b_s34b_b79k",
    lr=1e-3,
    epochs=1,
    use_contrastive=True,
):
    # Create pretrained CLIP
    clip_model, _, _ = open_clip.create_model_and_transforms(
        model_name, pretrained=pretrained
    )
    print("Model loaded:", model_name, pretrained)
    tokenizer = open_clip.get_tokenizer(model_name)

    clip_model = clip_model.to(device).eval()

    # Infer embedding dim safely by running one dummy pass
    with torch.no_grad():
        dummy_img = torch.zeros(1, 3, 224, 224, device=device)
        d = clip_model.encode_image(dummy_img).shape[-1]

    net = CLIPFiLMRMapper(clip_model, embed_dim=d).to(device)
    net.train()

    opt = torch.optim.AdamW(net.mapper.parameters(), lr=lr, weight_decay=1e-4)
    retrieval_history_original_emb_epoch = []
    retrieval_history_mapped_emb_epoch = []
    gaps_history_original_emb_epoch = {gap: [] for gap in gaps}
    gaps_history_mapped_emb_epoch = {gap: [] for gap in gaps}
    for epoch in tqdm(range(epochs), desc="Training (epochs)", leave=True):
        retrieval_history_original_emb_batch = []
        retrieval_history_mapped_emb_batch = []
        gaps_history_original_emb_batch = {gap: [] for gap in gaps}
        gaps_history_mapped_emb_batch = {gap: [] for gap in gaps}
        for step, batch in tqdm(
            enumerate(dataloader),
            total=len(dataloader) if hasattr(dataloader, "__len__") else None,
            desc=f"Epoch {epoch}",
            leave=False,
        ):
            images, captions, _, _ = batch

            images = images.to(device, non_blocking=True)

            # captions from Flickr30k are often a list[str] (or list of lists).
            # Make it robust:
            if isinstance(captions, (list, tuple)):
                # If captions are lists-of-lists, pick the first caption per image
                captions = [c[0] if isinstance(c, (list, tuple)) else c for c in captions]

            text_tokens = tokenizer(captions).to(device, non_blocking=True)

            z_hat, z_txt = net(images, text_tokens)

            if use_contrastive:
                loss = contrastive_loss(z_hat, z_txt)
            else:
                loss = cosine_regression_loss(z_hat, z_txt)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            
            # Update retrieval and gaps history
            with torch.no_grad():
                retrieval_history_original_emb_batch.append(retrieval(z_txt, z_txt))
                retrieval_history_mapped_emb_batch.append(retrieval(z_hat, z_txt))
                for gap in gaps:
                    gaps_history_original_emb_batch[gap].append(compute_gap(cf,gap,z_txt, z_txt,None)["text_vision"])
                    gaps_history_mapped_emb_batch[gap].append(compute_gap(cf,gap,z_hat, z_txt,None)["text_vision"])
                    
                if step % 50 == 0:
                    print(f"epoch {epoch} step {step} loss {loss.item():.4f}")
                    print(f"  retrieval original_emb: {retrieval_history_original_emb_batch[-1]:.4f}, mapped_emb: {retrieval_history_mapped_emb_batch[-1]:.4f}")
                    for gap in gaps:
                        #print the most recent gap metrics for this batch
                        print(f"  gap {gap} original_emb: {gaps_history_original_emb_batch[gap][-1]:.4f}, mapped_emb: {gaps_history_mapped_emb_batch[gap][-1]:.4f}")
                    
        
        # End of epoch: update epoch-level history
        
        retrieval_history_original_emb_epoch.append(np.mean(retrieval_history_original_emb_batch))
        retrieval_history_mapped_emb_epoch.append(np.mean(retrieval_history_mapped_emb_batch))
        for gap in gaps:
            gaps_history_original_emb_epoch[gap].append(np.mean(gaps_history_original_emb_batch[gap]))
            gaps_history_mapped_emb_epoch[gap].append(np.mean(gaps_history_mapped_emb_batch[gap]))
            
        print(f"End of epoch {epoch}: retrieval original_emb: {retrieval_history_original_emb_epoch[-1]:.4f}, mapped_emb: {retrieval_history_mapped_emb_epoch[-1]:.4f}")
        for gap in gaps:
            print(f"  gap {gap} original_emb: {gaps_history_original_emb_epoch[gap][-1]:.4f}, mapped_emb: {gaps_history_mapped_emb_epoch[gap][-1]:.4f}")
            
    
            
            

    return net

In [30]:
cf = load_configs_from_dir("../config_dir/test")[0][1]
train_loader, _, _ = get_dataloaders(cf)
net = train_filmr_mapper(cf, train_loader, device=device, epochs=1)

batch size: 256
Images (unique) in splits:
  train: 25426 val: 317 test: 6040
Pairs (image,caption) in splits (x5 per image):
  train: 127130 val: 1585 test: 6040
Model loaded: ViT-B-32 laion2b_s34b_b79k


Training (epochs):   0%|          | 0/1 [00:00<?, ?it/s]

epoch 0 step 0 loss 11.2846
  retrieval original_emb: 1.0000, mapped_emb: 0.0078
  gap RMG original_emb: -0.0000, mapped_emb: 0.8455
  gap L2M original_emb: 0.0000, mapped_emb: 0.7392
  gap L2I original_emb: 0.0000, mapped_emb: 1.4089


Training (epochs):   0%|          | 0/1 [00:02<?, ?it/s]


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.